In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/2023-10-12-Image-Master-BTW21.csv')

In [ ]:
df.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,image,bucket_url,vertex_caption,caption,type,Faces,OCR,People,Objects,Classification,Multi Classification
0,0,0,0,0,abaerbock_25-09-2021_00:00_image2.jpeg,abaerbock_25-09-2021_00:00_image2.jpeg,a group of people wearing masks and holding si...,a group of people holding signs,Story,> 9,Und deswegen gehen heute wieder in vielen Städ...,5,3x Top,Issue-Based Messaging,"['Campaign Activities', 'Public Engagement', '..."
1,1,1,1,1,olafscholz_22-09-2021_00:00_video10.jpeg,olafscholz_22-09-2021_00:00_video10.jpeg,a man wearing a mask is standing in the dark,Olaf Scholz in a suit and tie,Story,2.0,Und natürlich dürfen selbst im Dunkeln die Sel...,2,"2x Outerwear, 1x Clothing",Politician Portrayals,"['Politician Portrayals', 'Public Engagement']"
2,2,2,2,2,abaerbock_13-09-2021_00:00_video3.jpeg,abaerbock_13-09-2021_00:00_video3.jpeg,a group of people standing in front of a sign ...,graphical user interface,Story,0,Hier wird gleich live das Triell gesendet. An ...,6,4x Clothing,Traditional Media Campaigning,"['Traditional Media Campaigning', 'Public Enga..."
3,3,3,3,3,cdu_20-09-2021_00:00_video24.jpeg,cdu_20-09-2021_00:00_video24.jpeg,a man in a suit is talking to a woman in a gre...,a man and woman,Story,3.0,Was in #Afghanistan passierte; darf nicht noch...,5,"1x Television, 1x Outerwear, 1x Clothing, 1x Top",Public Engagement,"['Public Engagement', 'Traditional Media Campa..."
4,4,4,4,4,cdu_24-09-2021_00:00_video3.jpeg,cdu_24-09-2021_00:00_video3.jpeg,a woman is smiling in front of a stone wall an...,a woman smiling for the camera,Story,1.0,& ist 2) 2021 @catarinadlssu?s_ 'tatsid Esciel...,1,"1x Outerwear, 1x Clothing",Politician Portrayals,['Politician Portrayals']


Ich möchte alle Posts extrahieren und darüber Gesichter erkennen lassen, um sie von Menschen annotieren zu lassen.

In [ ]:
df_filtered = df[df['type'] == "Post"]

In [ ]:
df_filtered.head()

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,image,bucket_url,vertex_caption,caption,type,Faces,OCR,People,Objects,Classification,Multi Classification
2208,2208,2208,2208,2208,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,a man in a suit shakes hands with two women,a group of people standing in a room,Post,4.0,NaN,3,"4x Pants, 3x Outerwear",Public Engagement,['Public Engagement']
2209,2209,2209,2209,2209,armin_laschet/2021-09-18_19-12-07_UTC_5.jpg,armin_laschet/2021-09-18_19-12-07_UTC_5.jpg,a group of people posing for a picture with a ...,"Hermann Gröhe, Armin Laschet et al. posing for...",Post,1.0,Oa FOTO Ko CIEE,6,"3x Glasses, 1x Tie",Politician Portrayals,NaN
2210,2210,2210,2210,2210,armin_laschet/2021-09-26_23-29-21_UTC_3.jpg,armin_laschet/2021-09-26_23-29-21_UTC_3.jpg,a group of people standing in front of a wall ...,Armin Laschet et al. standing in front of a flag,Post,0,NaN,0,NaN,Campaign Activities,['Politician Portrayals']
2211,2211,2211,2211,2211,armin_laschet/2021-09-15_12-38-48_UTC.jpg,armin_laschet/2021-09-15_12-38-48_UTC.jpg,a man in a suit and tie stands in front of a s...,Armin Laschet in a suit and tie,Post,1.0,ARD-Wahlarena Mittwoch 20.15 Uhr ARD 8 1 :,1,"1x Tie, 1x Glasses",Traditional Media Campaigning,['Traditional Media Campaigning']
2212,2212,2212,2212,2212,armin_laschet/2021-09-21_11-03-21_UTC_1.jpg,armin_laschet/2021-09-21_11-03-21_UTC_1.jpg,a man in a suit and tie is giving a speech,Armin Laschet in a suit speaking into a microp...,Post,3.0,Durch das schnelle und konsequente Handeln ko...,1,1x Tie,Campaign Activities,NaN


Extrahiere noch die Bilder.

In [ ]:
!unzip ../data/posts.zip -d images

In [ ]:
df_filtered = df_filtered[["image", "type", "bucket_url"]]

In [ ]:
df_filtered['image'] = df_filtered['image'].apply(lambda x: f"images/{x}")

In [ ]:
df_filtered.head()

,image,type,bucket_url
2208,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg
2209,images/armin_laschet/2021-09-18_19-12-07_UTC_5...,Post,armin_laschet/2021-09-18_19-12-07_UTC_5.jpg
2210,images/armin_laschet/2021-09-26_23-29-21_UTC_3...,Post,armin_laschet/2021-09-26_23-29-21_UTC_3.jpg
2211,images/armin_laschet/2021-09-15_12-38-48_UTC.jpg,Post,armin_laschet/2021-09-15_12-38-48_UTC.jpg
2212,images/armin_laschet/2021-09-21_11-03-21_UTC_1...,Post,armin_laschet/2021-09-21_11-03-21_UTC_1.jpg


Führe Bilderkennung aus

In [ ]:
!pip install deepface
!pip install retina-face

In [ ]:
#@title ### Detect all faces using RetinaFace

# Required Libraries
from retinaface import RetinaFace
from tqdm.auto import tqdm
import uuid
import pandas as pd  # Assuming you're using a pandas DataFrame
import logging

all_faces = []  # Contains a dict for each face found

for index, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    faces_found = ""

    try:
        faces_found = RetinaFace.detect_faces(row['image'])
    except FileNotFoundError:
        logging.warning(f"File not found: {row['image']}")
        continue  # Skip this iteration and proceed with the next image
    except Exception as e:
        logging.error(f"Error processing file {row['image']}: {e}")
        continue

    # In case of no face in an image, RetinaFace returns a tuple; otherwise, a dict
    if isinstance(faces_found, dict):
        for key, value in faces_found.items():
            face = {
                **row.to_dict(),
                "face_uuid": uuid.uuid4(),
                "retina_face_score": value['score'],
                "facial_area": value['facial_area'],
                "right_eye": value['landmarks']['right_eye'],
                "left_eye": value['landmarks']['left_eye'],
                "nose": value['landmarks']['nose'],
                "mouth_right": value['landmarks']['mouth_right'],
                "mouth_left": value['landmarks']['mouth_left']
            }

            all_faces.append(face)

# Creates a pandas DataFrame for all faces and the number of faces per image
# The latter can then be merged into the initial DataFrame using the common column "filename"
all_faces_df = pd.DataFrame.from_dict(all_faces)

export_filepath = '../data/'  # @param {type: "string"}
filename_all_faces_df = '2024-03-18-Retina-Face-Posts-Detection'  # @param {type: "string"}

# Saves the DataFrame to a CSV file in your drive
all_faces_df.to_csv(f'{export_filepath}{filename_all_faces_df}.csv', index=False)

In [ ]:
all_faces_df.head()

,image,type,bucket_url,face_uuid,retina_face_score,facial_area,right_eye,left_eye,nose,mouth_right,mouth_left
0,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,1e19a944-ce54-4a50-af5f-1fb26ae46852,0.999463,"[618, 68, 698, 179]","[655.83215, 115.59756]","[687.78125, 116.61998]","[679.88385, 137.47713]","[651.548, 148.28584]","[682.7433, 149.67479]"
1,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,7c50af0d-d49f-4121-921b-173b16eb9a63,0.999458,"[865, 76, 941, 176]","[882.80505, 117.19016]","[918.87573, 113.6915]","[901.9659, 135.02817]","[887.5133, 148.83932]","[923.07104, 145.7976]"
2,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,f0902030-6596-4e02-a702-9ac7c47c5e16,0.999433,"[217, 96, 293, 203]","[247.93494, 139.99324]","[282.4035, 137.506]","[276.23734, 155.95622]","[253.7339, 175.65933]","[284.2945, 173.6826]"
3,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,d4d4e4e0-f05f-47f5-8dc3-754335fd3e62,0.995322,"[418, 111, 460, 165]","[427.0738, 133.0951]","[445.76318, 133.36816]","[434.90997, 144.35779]","[429.42117, 153.67068]","[442.7721, 153.90323]"
4,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,armin_laschet/2021-09-22_14-55-06_UTC_2.jpg,b57bfd65-c875-4f24-9273-ed43955386d2,0.995100,"[986, 65, 1075, 227]","[1001.7438, 125.714874]","[1003.1344, 126.80907]","[987.33704, 157.53435]","[1007.1196, 187.25154]","[1009.04364, 187.13972]"



## Export to LabelStudio

In [ ]:
import cv2
import numpy as np
import pandas as pd

def getImageDimensions(path):
    """Returns both width and height of an image at the given path."""
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is not None:
        height, width = img.shape[:2]
        return width, height
    else:
        return np.nan, np.nan

In [ ]:
# Assuming all_faces_df is your DataFrame and it has an 'image' column with paths to the images
# Update the function to handle both dimensions at once to minimize I/O operations
from tqdm.notebook import tqdm
tqdm.pandas()


# Create a temporary DataFrame with the new dimensions
dimensions_df = all_faces_df['image'].progress_apply(lambda x: pd.Series(getImageDimensions(x), index=['image_width', 'image_height']))

# Join this temporary DataFrame back to your original DataFrame
all_faces_df = pd.concat([all_faces_df, dimensions_df], axis=1)

In [ ]:
all_faces_df['bucket_url'] = all_faces_df['bucket_url'].apply(lambda x: f"../data/{x}")

In [ ]:
all_faces_df.head()

,image,type,bucket_url,face_uuid,retina_face_score,facial_area,right_eye,left_eye,nose,mouth_right,mouth_left,image_width,image_height
0,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,1e19a944-ce54-4a50-af5f-1fb26ae46852,0.999463,"[618, 68, 698, 179]","[655.83215, 115.59756]","[687.78125, 116.61998]","[679.88385, 137.47713]","[651.548, 148.28584]","[682.7433, 149.67479]",1440,856
1,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,7c50af0d-d49f-4121-921b-173b16eb9a63,0.999458,"[865, 76, 941, 176]","[882.80505, 117.19016]","[918.87573, 113.6915]","[901.9659, 135.02817]","[887.5133, 148.83932]","[923.07104, 145.7976]",1440,856
2,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,f0902030-6596-4e02-a702-9ac7c47c5e16,0.999433,"[217, 96, 293, 203]","[247.93494, 139.99324]","[282.4035, 137.506]","[276.23734, 155.95622]","[253.7339, 175.65933]","[284.2945, 173.6826]",1440,856
3,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,d4d4e4e0-f05f-47f5-8dc3-754335fd3e62,0.995322,"[418, 111, 460, 165]","[427.0738, 133.0951]","[445.76318, 133.36816]","[434.90997, 144.35779]","[429.42117, 153.67068]","[442.7721, 153.90323]",1440,856
4,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,b57bfd65-c875-4f24-9273-ed43955386d2,0.995100,"[986, 65, 1075, 227]","[1001.7438, 125.714874]","[1003.1344, 126.80907]","[987.33704, 157.53435]","[1007.1196, 187.25154]","[1009.04364, 187.13972]",1440,856


In [ ]:
# Coordiantes of the upper left corner
all_faces_df['f_pixel_x'] = all_faces_df.apply(lambda x: x['facial_area'][0], axis=1)
all_faces_df['f_pixel_y'] = all_faces_df.apply(lambda x: x['facial_area'][1], axis=1)

# width and height of each face in pixel
all_faces_df['f_pixel_width'] = all_faces_df.apply(lambda x: int(x['facial_area'][2]) - int(x['facial_area'][0]), axis=1)
all_faces_df['f_pixel_height'] = all_faces_df.apply(lambda x: int(x['facial_area'][3]) - int(x['facial_area'][1]), axis=1)

In [ ]:
all_faces_df['f_relativ_x'] = all_faces_df.apply(lambda x: x['f_pixel_x'] / x['image_width'] * 100, axis=1)
all_faces_df['f_relativ_y'] = all_faces_df.apply(lambda x:  x['f_pixel_y'] / x['image_height'] * 100, axis=1)
all_faces_df['f_relativ_width'] = all_faces_df.apply(lambda x: x['f_pixel_width'] / x['image_width'] * 100, axis=1)
all_faces_df['f_relativ_height'] = all_faces_df.apply(lambda x: x['f_pixel_height'] / x['image_height'] * 100, axis=1)

In [ ]:
all_faces_df.head()

,image,type,bucket_url,face_uuid,retina_face_score,facial_area,right_eye,left_eye,nose,mouth_right,...,image_width,image_height,f_pixel_x,f_pixel_y,f_pixel_width,f_pixel_height,f_relativ_x,f_relativ_y,f_relativ_width,f_relativ_height
0,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,1e19a944-ce54-4a50-af5f-1fb26ae46852,0.999463,"[618, 68, 698, 179]","[655.83215, 115.59756]","[687.78125, 116.61998]","[679.88385, 137.47713]","[651.548, 148.28584]",...,1440,856,618,68,80,111,42.916667,7.943925,5.555556,12.967290
1,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,7c50af0d-d49f-4121-921b-173b16eb9a63,0.999458,"[865, 76, 941, 176]","[882.80505, 117.19016]","[918.87573, 113.6915]","[901.9659, 135.02817]","[887.5133, 148.83932]",...,1440,856,865,76,76,100,60.069444,8.878505,5.277778,11.682243
2,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,f0902030-6596-4e02-a702-9ac7c47c5e16,0.999433,"[217, 96, 293, 203]","[247.93494, 139.99324]","[282.4035, 137.506]","[276.23734, 155.95622]","[253.7339, 175.65933]",...,1440,856,217,96,76,107,15.069444,11.214953,5.277778,12.500000
3,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,d4d4e4e0-f05f-47f5-8dc3-754335fd3e62,0.995322,"[418, 111, 460, 165]","[427.0738, 133.0951]","[445.76318, 133.36816]","[434.90997, 144.35779]","[429.42117, 153.67068]",...,1440,856,418,111,42,54,29.027778,12.967290,2.916667,6.308411
4,images/armin_laschet/2021-09-22_14-55-06_UTC_2...,Post,gs://ig-politik-posts/armin_laschet/2021-09-22...,b57bfd65-c875-4f24-9273-ed43955386d2,0.995100,"[986, 65, 1075, 227]","[1001.7438, 125.714874]","[1003.1344, 126.80907]","[987.33704, 157.53435]","[1007.1196, 187.25154]",...,1440,856,986,65,89,162,68.472222,7.593458,6.180556,18.925234


In [ ]:
all_faces_df.to_csv(f'{export_filepath}{filename_all_faces_df}.csv', index=False)

In [ ]:
#@title ## Build json prediction
#@markdown Builds a "prediction" in JSON format for a single face

def buildPredictionJSON(face_id):

  face = all_faces_df.loc[all_faces_df['face_uuid'] == face_id ]

  face_data = {
    "original_width": face['image_width'].item(),
    "original_height": face['image_height'].item(),
    "image_rotation": 0,
    "value": {
        "x": face['f_relativ_x'].item(),
        "y": face['f_relativ_y'].item(),
        "width": face['f_relativ_width'].item(),
        "height": face['f_relativ_height'].item(),
        "rotation": 0
    },
    "id": str(face['face_uuid'].item()),
    "from_name": "rect",
    "to_name": "image",
    "type": "rectangle"
    }

  return face_data

In [ ]:
import json

def buildTaskJSON(image):
    # Filter rows for the given image
    image_rows = all_faces_df[all_faces_df['image'] == image]

    # Return immediately if no rows are found for the image
    if image_rows.empty:
        return None  # Return an empty JSON array or handle as needed

    # Since all rows in image_rows pertain to the same image, we can safely assume all have the same bucket_url
    bucket_url = image_rows.iloc[0]['bucket_url']

    # Extract face_ids for the given image
    face_ids = image_rows['face_uuid'].tolist()

    # Construct tasks list
    tasks_list = {
        "data": {
            "image": bucket_url
        },
        "predictions": [{
            "model_version": "RetinaFace",
            "result": [buildPredictionJSON(face_id) for face_id in face_ids]
        }]
    }

    return json.dumps(tasks_list)

In [ ]:
images = all_faces_df['image'].unique().tolist()

tasks = []

for image in tqdm(images):
  task = buildTaskJSON(image)
  if task:
    tasks.append(task)

In [ ]:
len(tasks)

1011

In [ ]:
!rm -r /content/tasks

rm: cannot remove '/content/tasks': No such file or directory


In [ ]:
import os

# Assuming you have a directory path where you want to save the JSON files
output_dir = "/content/tasks"
os.makedirs(output_dir, exist_ok=True)  # Create the output directory if it doesn't exist

for index, task in enumerate(tqdm(tasks)):
  file_name = f"deepface_v7_task_{index}.json"  # Example filename, consider using parts of the image name
  file_path = os.path.join(output_dir, file_name)

        # Write the task JSON to a file
  with open(file_path, 'w') as f:
    f.write(task)

In [ ]:
!zip -r "../data/2024-03-18-Posts-Face-Tasks.zip" "/content/tasks"

In [ ]:
# Install Google Cloud Storage library
!pip install google-cloud-storage

from google.colab import auth
auth.authenticate_user()

from google.cloud import storage
from oauth2client.client import GoogleCredentials
import os

# Assuming your output directory
output_dir = "/content/tasks"

# Authenticate this Colab notebook with your Google Account
credentials = GoogleCredentials.get_application_default()

# Initialize the Google Cloud Storage client with your credentials
client = storage.Client(credentials=credentials, project="# REMOVED: GCP service account key not needed for local execution")

# The name of your Google Cloud Storage bucket
bucket_name = 'ig-politik-posts'

# Get the bucket object
bucket = client.get_bucket(bucket_name)

# List all files in the specified directory and upload each to the GCS bucket
for filename in os.listdir(output_dir):
    file_path = os.path.join(output_dir, filename)

    # Create a blob object
    blob = bucket.blob(filename)

    # Upload the file to Google Cloud Storage
    blob.upload_from_filename(file_path)

    print(f'Uploaded {filename} to "tasks/" directory in bucket {bucket_name}')